# Indian Number Plate Detection + OCR Notebook  
## Customized for the DataCluster Labs Kaggle dataset

This notebook is tailored for the **Indian Number Plates Dataset** from DataCluster Labs on Kaggle.

It supports the annotation formats this dataset family commonly provides:

- YOLO
- COCO
- Pascal VOC

and converts them into one standard training-ready structure.

---

## What this notebook does

### Stage 1 — Detection
Train a number plate detector using the Kaggle dataset.

### Stage 2 — OCR
Create cropped plate images and train a custom OCR model to read the plate text.

### Stage 3 — Inference
Use the trained detector + OCR model together to get the plate number from new images.

---

## Practical note

This dataset is strongest for **detection**.  
For **OCR text recognition**, you may still need to prepare or verify the plate text labels for each cropped plate image, depending on what exactly is included in the package you downloaded.

So this notebook includes:

- direct detector training
- automatic conversion from multiple annotation formats
- cropped plate generation
- OCR dataset creation workflow
- OCR training
- final end-to-end prediction

In [ ]:

# =========================
# 1. Install dependencies
# =========================

!pip install -q ultralytics opencv-python matplotlib pandas numpy pillow scikit-learn tqdm pyyaml lxml

## 2. Imports

In [1]:

import os
import cv2
import yaml
import json
import math
import shutil
import random
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from ultralytics import YOLO

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Using device:", DEVICE)

Torch: 2.3.1+cu118
CUDA available: True
Using device: cuda


## 3. Configuration

Set the location of the extracted Kaggle dataset here.

In [2]:

# =========================
# 3. Configuration
# =========================

RAW_DATASET_DIR = Path("./raw_indian_number_plates_dataset")   # <- change this
PROJECT_ROOT = Path("./indian_plate_project")
WORK_DIR = PROJECT_ROOT / "prepared_data"
RUNS_DIR = PROJECT_ROOT / "runs"

WORK_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

DETECTOR_DIR = WORK_DIR / "detector_dataset"
OCR_DIR = WORK_DIR / "ocr_dataset"

DETECTOR_MODEL = "yolov8n.pt"
DETECTOR_IMG_SIZE = 640
DETECTOR_BATCH = 16
DETECTOR_EPOCHS = 50

OCR_IMG_H = 64
OCR_IMG_W = 256
OCR_BATCH_SIZE = 32
OCR_EPOCHS = 25
OCR_LR = 1e-3

CHAR_SET = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"
BLANK_IDX = 0
CHAR_TO_IDX = {c: i + 1 for i, c in enumerate(CHAR_SET)}
IDX_TO_CHAR = {i + 1: c for i, c in enumerate(CHAR_SET)}

print("RAW_DATASET_DIR =", RAW_DATASET_DIR.resolve())
print("PROJECT_ROOT =", PROJECT_ROOT.resolve())

RAW_DATASET_DIR = C:\Users\Deep\Desktop\DAU_Projects\traffic\number_plate_detection\raw_indian_number_plates_dataset
PROJECT_ROOT = C:\Users\Deep\Desktop\DAU_Projects\traffic\number_plate_detection\indian_plate_project


## 4. Explore the dataset

In [3]:

def print_tree(path, max_depth=3, indent=""):
    path = Path(path)
    if not path.exists():
        print(f"[MISSING] {path}")
        return
    print(indent + path.name)
    if max_depth == 0:
        return
    for child in sorted(path.iterdir()):
        if child.is_dir():
            print_tree(child, max_depth=max_depth - 1, indent=indent + "  ")
        else:
            print(indent + "  " + child.name)

print_tree(RAW_DATASET_DIR, max_depth=3)

[MISSING] raw_indian_number_plates_dataset


## 5. Detect available annotation style

In [ ]:

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def list_files(root, suffixes=None):
    root = Path(root)
    files = []
    for p in root.rglob("*"):
        if p.is_file():
            if suffixes is None or p.suffix.lower() in suffixes:
                files.append(p)
    return files

image_files = list_files(RAW_DATASET_DIR, IMG_EXTS)
txt_files = list_files(RAW_DATASET_DIR, {".txt"})
xml_files = list_files(RAW_DATASET_DIR, {".xml"})
json_files = list_files(RAW_DATASET_DIR, {".json"})

print("Images found:", len(image_files))
print("TXT found   :", len(txt_files))
print("XML found   :", len(xml_files))
print("JSON found  :", len(json_files))

def find_coco_json_candidates(json_files):
    candidates = []
    for jf in json_files:
        try:
            with open(jf, "r", encoding="utf-8") as f:
                data = json.load(f)
            if isinstance(data, dict) and "images" in data and "annotations" in data:
                candidates.append(jf)
        except Exception:
            pass
    return candidates

coco_json_candidates = find_coco_json_candidates(json_files)
image_stems = {p.stem for p in image_files}
matching_yolo_like = [p for p in txt_files if p.stem in image_stems]

annotation_mode = None
if len(matching_yolo_like) > 0:
    annotation_mode = "yolo"
elif len(coco_json_candidates) > 0:
    annotation_mode = "coco"
elif len(xml_files) > 0:
    annotation_mode = "voc"

print("Detected annotation mode:", annotation_mode)

## 6. Standardization helpers

In [ ]:

def make_clean_dir(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

def split_list(items, train_ratio=0.7, val_ratio=0.2, seed=42):
    items = list(items)
    random.Random(seed).shuffle(items)
    n = len(items)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    train_items = items[:n_train]
    val_items = items[n_train:n_train+n_val]
    test_items = items[n_train+n_val:]
    return train_items, val_items, test_items

def copy_pair(img_path, label_text, split, out_root):
    img_out = out_root / "images" / split / img_path.name
    lbl_out = out_root / "labels" / split / f"{img_path.stem}.txt"
    img_out.parent.mkdir(parents=True, exist_ok=True)
    lbl_out.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(img_path, img_out)
    with open(lbl_out, "w", encoding="utf-8") as f:
        f.write(label_text)

def to_yolo_line(xmin, ymin, xmax, ymax, img_w, img_h, cls_id=0):
    xc = ((xmin + xmax) / 2.0) / img_w
    yc = ((ymin + ymax) / 2.0) / img_h
    bw = (xmax - xmin) / img_w
    bh = (ymax - ymin) / img_h
    return f"{cls_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}"

## 7. Prepare YOLO-format detector dataset

In [ ]:

def prepare_from_yolo(raw_root, out_root, seed=42):
    raw_root = Path(raw_root)
    out_root = Path(out_root)
    make_clean_dir(out_root)

    images = [p for p in raw_root.rglob("*") if p.suffix.lower() in IMG_EXTS]
    pairs = []
    for img in images:
        txt = img.with_suffix(".txt")
        if txt.exists():
            pairs.append((img, txt))

    print("YOLO pairs found:", len(pairs))
    if len(pairs) == 0:
        raise ValueError("No YOLO image-label pairs found.")

    train_pairs, val_pairs, test_pairs = split_list(pairs, 0.7, 0.2, seed)

    for split_name, split_pairs in [("train", train_pairs), ("val", val_pairs), ("test", test_pairs)]:
        for img, txt in tqdm(split_pairs, desc=f"Copy {split_name}"):
            (out_root / "images" / split_name).mkdir(parents=True, exist_ok=True)
            (out_root / "labels" / split_name).mkdir(parents=True, exist_ok=True)
            shutil.copy2(img, out_root / "images" / split_name / img.name)
            shutil.copy2(txt, out_root / "labels" / split_name / f"{img.stem}.txt")

## 8. Prepare from COCO

In [ ]:

def find_best_coco_json(candidates):
    if len(candidates) == 0:
        return None
    ranked = sorted(
        candidates,
        key=lambda p: (
            "instances" not in p.name.lower(),
            "annotation" not in str(p).lower(),
            len(str(p))
        )
    )
    return ranked[0]

def prepare_from_coco(coco_json_path, raw_root, out_root, seed=42):
    raw_root = Path(raw_root)
    out_root = Path(out_root)
    make_clean_dir(out_root)

    with open(coco_json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    images_info = {img["id"]: img for img in data["images"]}
    anns_by_image = defaultdict(list)
    for ann in data["annotations"]:
        anns_by_image[ann["image_id"]].append(ann)

    image_records = []
    for image_id, img_info in images_info.items():
        file_name = img_info["file_name"]
        img_path = raw_root / file_name
        if not img_path.exists():
            candidates = list(raw_root.rglob(Path(file_name).name))
            if len(candidates) == 0:
                continue
            img_path = candidates[0]
        image_records.append((image_id, img_path, img_info))

    train_recs, val_recs, test_recs = split_list(image_records, 0.7, 0.2, seed)

    for split_name, split_recs in [("train", train_recs), ("val", val_recs), ("test", test_recs)]:
        for image_id, img_path, img_info in tqdm(split_recs, desc=f"COCO {split_name}"):
            w = img_info["width"]
            h = img_info["height"]
            yolo_lines = []
            for ann in anns_by_image.get(image_id, []):
                x, y, bw, bh = ann["bbox"]
                xmin, ymin, xmax, ymax = x, y, x + bw, y + bh
                yolo_lines.append(to_yolo_line(xmin, ymin, xmax, ymax, w, h, cls_id=0))
            copy_pair(img_path, "\n".join(yolo_lines), split_name, out_root)

## 9. Prepare from Pascal VOC

In [ ]:

def parse_voc_xml(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    filename = root.findtext("filename")
    size = root.find("size")
    width = int(size.findtext("width"))
    height = int(size.findtext("height"))

    boxes = []
    for obj in root.findall("object"):
        bnd = obj.find("bndbox")
        xmin = float(bnd.findtext("xmin"))
        ymin = float(bnd.findtext("ymin"))
        xmax = float(bnd.findtext("xmax"))
        ymax = float(bnd.findtext("ymax"))
        boxes.append((xmin, ymin, xmax, ymax))

    return filename, width, height, boxes

def prepare_from_voc(raw_root, out_root, seed=42):
    raw_root = Path(raw_root)
    out_root = Path(out_root)
    make_clean_dir(out_root)

    xmls = list(raw_root.rglob("*.xml"))
    records = []

    for xmlp in xmls:
        try:
            filename, width, height, boxes = parse_voc_xml(xmlp)
            img_path = xmlp.parent / filename
            if not img_path.exists():
                candidates = list(raw_root.rglob(filename))
                if len(candidates) == 0:
                    continue
                img_path = candidates[0]
            records.append((img_path, width, height, boxes))
        except Exception:
            pass

    train_recs, val_recs, test_recs = split_list(records, 0.7, 0.2, seed)

    for split_name, split_recs in [("train", train_recs), ("val", val_recs), ("test", test_recs)]:
        for img_path, w, h, boxes in tqdm(split_recs, desc=f"VOC {split_name}"):
            yolo_lines = [to_yolo_line(xmin, ymin, xmax, ymax, w, h, cls_id=0) for xmin, ymin, xmax, ymax in boxes]
            copy_pair(img_path, "\n".join(yolo_lines), split_name, out_root)

## 10. Run the dataset preparation

In [ ]:

if annotation_mode == "yolo":
    prepare_from_yolo(RAW_DATASET_DIR, DETECTOR_DIR)
elif annotation_mode == "coco":
    best_coco_json = find_best_coco_json(coco_json_candidates)
    print("Using COCO file:", best_coco_json)
    prepare_from_coco(best_coco_json, RAW_DATASET_DIR, DETECTOR_DIR)
elif annotation_mode == "voc":
    prepare_from_voc(RAW_DATASET_DIR, DETECTOR_DIR)
else:
    raise ValueError("No supported annotations found.")

for split in ["train", "val", "test"]:
    img_count = len(list((DETECTOR_DIR / "images" / split).glob("*.*")))
    lbl_count = len(list((DETECTOR_DIR / "labels" / split).glob("*.txt")))
    print(split, "images =", img_count, "| labels =", lbl_count)

## 11. Visualize prepared labels

In [ ]:

def yolo_to_xyxy(label_line, img_w, img_h):
    cls, xc, yc, bw, bh = map(float, label_line.strip().split())
    x1 = int((xc - bw / 2) * img_w)
    y1 = int((yc - bh / 2) * img_h)
    x2 = int((xc + bw / 2) * img_w)
    y2 = int((yc + bh / 2) * img_h)
    return int(cls), x1, y1, x2, y2

def show_detector_samples(images_dir, labels_dir, n=6):
    images = list(Path(images_dir).glob("*.*"))
    if len(images) == 0:
        print("No images found.")
        return
    sample = random.sample(images, min(n, len(images)))
    cols = 3
    rows = math.ceil(len(sample) / cols)
    plt.figure(figsize=(15, 5 * rows))
    for i, img_path in enumerate(sample, 1):
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        label_path = Path(labels_dir) / f"{img_path.stem}.txt"
        if label_path.exists():
            for line in label_path.read_text().strip().splitlines():
                if line.strip():
                    _, x1, y1, x2, y2 = yolo_to_xyxy(line, w, h)
                    cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
        plt.subplot(rows, cols, i)
        plt.imshow(img)
        plt.title(img_path.name)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

show_detector_samples(DETECTOR_DIR / "images/train", DETECTOR_DIR / "labels/train", n=6)

## 12. Create detector YAML

In [ ]:

detector_yaml_path = WORK_DIR / "detector_data.yaml"

detector_yaml = {
    "path": str(DETECTOR_DIR.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {0: "number_plate"}
}

with open(detector_yaml_path, "w") as f:
    yaml.dump(detector_yaml, f)

print(detector_yaml_path.read_text())

## 13. Train the detector

In [ ]:

detector_model = YOLO(DETECTOR_MODEL)

detector_results = detector_model.train(
    data=str(detector_yaml_path),
    epochs=DETECTOR_EPOCHS,
    imgsz=DETECTOR_IMG_SIZE,
    batch=DETECTOR_BATCH,
    project=str(RUNS_DIR),
    name="indian_plate_detector",
    exist_ok=True
)

## 14. Validate detector

In [ ]:

best_detector_path = RUNS_DIR / "indian_plate_detector" / "weights" / "best.pt"
detector_model = YOLO(str(best_detector_path))
metrics = detector_model.val(data=str(detector_yaml_path))
print(metrics)

## 15. Show detector predictions

In [ ]:

test_imgs = list((DETECTOR_DIR / "images/test").glob("*.*"))[:6]

plt.figure(figsize=(16, 10))
for i, img_path in enumerate(test_imgs, 1):
    res = detector_model(str(img_path), conf=0.25)
    plotted = res[0].plot()
    plotted = cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB)
    plt.subplot(2, 3, i)
    plt.imshow(plotted)
    plt.title(img_path.name)
    plt.axis("off")
plt.tight_layout()
plt.show()

# OCR preparation

This part crops plate regions from the labeled detector dataset.

Then you fill the plate text values into a CSV and train OCR.

## 16. Create cropped plates + OCR template CSV

In [ ]:

def save_gt_plate_crops(detector_dir, ocr_dir):
    detector_dir = Path(detector_dir)
    ocr_dir = Path(ocr_dir)

    if ocr_dir.exists():
        shutil.rmtree(ocr_dir)
    for split in ["train", "val", "test"]:
        (ocr_dir / "images" / split).mkdir(parents=True, exist_ok=True)

    rows = []

    for split in ["train", "val", "test"]:
        img_dir = detector_dir / "images" / split
        lbl_dir = detector_dir / "labels" / split

        for img_path in tqdm(list(img_dir.glob("*.*")), desc=f"Cropping {split}"):
            label_path = lbl_dir / f"{img_path.stem}.txt"
            if not label_path.exists():
                continue

            img_bgr = cv2.imread(str(img_path))
            if img_bgr is None:
                continue
            h, w = img_bgr.shape[:2]

            lines = label_path.read_text().strip().splitlines()
            for j, line in enumerate(lines):
                if not line.strip():
                    continue
                _, x1, y1, x2, y2 = yolo_to_xyxy(line, w, h)
                x1 = max(0, x1); y1 = max(0, y1); x2 = min(w, x2); y2 = min(h, y2)
                crop = img_bgr[y1:y2, x1:x2]
                if crop.size == 0:
                    continue
                crop_name = f"{img_path.stem}_plate_{j}.jpg"
                crop_path = ocr_dir / "images" / split / crop_name
                cv2.imwrite(str(crop_path), crop)

                rows.append({
                    "image_name": crop_name,
                    "text": "",
                    "split": split,
                    "source_image": img_path.name
                })

    labels_df = pd.DataFrame(rows)
    labels_df.to_csv(ocr_dir / "labels_template.csv", index=False)
    return labels_df

ocr_template_df = save_gt_plate_crops(DETECTOR_DIR, OCR_DIR)
print("Saved:", OCR_DIR / "labels_template.csv")
display(ocr_template_df.head())
print("Total OCR crops:", len(ocr_template_df))

## 17. Fill `labels.csv`

Open `labels_template.csv`, fill the `text` column with the real plate number, and save it as:

`ocr_dataset/labels.csv`

Use uppercase only, for example:
- GJ01AB1234
- MH12DE1433
- DL8CAF5031

In [ ]:

completed_labels_path = OCR_DIR / "labels.csv"

if completed_labels_path.exists():
    print("Found OCR labels:", completed_labels_path)
    ocr_df = pd.read_csv(completed_labels_path)
    display(ocr_df.head())
else:
    print("OCR labels.csv not found yet.")

## 18. OCR cleaning

In [ ]:

def clean_plate_text(text):
    text = str(text).upper().strip()
    allowed = set(CHAR_SET)
    return "".join([c for c in text if c in allowed])

if completed_labels_path.exists():
    ocr_df = pd.read_csv(completed_labels_path)
    ocr_df["text"] = ocr_df["text"].apply(clean_plate_text)
    ocr_df = ocr_df[ocr_df["text"].str.len() > 0].reset_index(drop=True)
    print("OCR rows after cleaning:", len(ocr_df))
    display(ocr_df.head())

## 19. OCR Dataset + DataLoader

In [ ]:

class OCRDataset(Dataset):
    def __init__(self, df, base_dir, img_h=64, img_w=256):
        self.df = df.reset_index(drop=True)
        self.base_dir = Path(base_dir)
        self.img_h = img_h
        self.img_w = img_w

    def __len__(self):
        return len(self.df)

    def encode_text(self, text):
        return [CHAR_TO_IDX[c] for c in text]

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.base_dir / row["split"] / row["image_name"]
        text = row["text"]

        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise FileNotFoundError(f"Could not read image: {img_path}")

        img = cv2.resize(img, (self.img_w, self.img_h))
        img = img.astype(np.float32) / 255.0
        img = np.expand_dims(img, axis=0)

        target = self.encode_text(text)
        return {
            "image": torch.tensor(img, dtype=torch.float32),
            "target": torch.tensor(target, dtype=torch.long),
            "target_text": text
        }

def ocr_collate_fn(batch):
    images = torch.stack([x["image"] for x in batch], dim=0)
    targets = [x["target"] for x in batch]
    target_texts = [x["target_text"] for x in batch]
    target_lengths = torch.tensor([len(t) for t in targets], dtype=torch.long)
    targets_concat = torch.cat(targets)
    return images, targets_concat, target_lengths, target_texts

if completed_labels_path.exists():
    train_df = ocr_df[ocr_df["split"] == "train"].copy()
    val_df = ocr_df[ocr_df["split"] == "val"].copy()
    test_df = ocr_df[ocr_df["split"] == "test"].copy()

    train_ds = OCRDataset(train_df, OCR_DIR / "images", OCR_IMG_H, OCR_IMG_W)
    val_ds = OCRDataset(val_df, OCR_DIR / "images", OCR_IMG_H, OCR_IMG_W)
    test_ds = OCRDataset(test_df, OCR_DIR / "images", OCR_IMG_H, OCR_IMG_W)

    train_loader = DataLoader(train_ds, batch_size=OCR_BATCH_SIZE, shuffle=True, collate_fn=ocr_collate_fn)
    val_loader = DataLoader(val_ds, batch_size=OCR_BATCH_SIZE, shuffle=False, collate_fn=ocr_collate_fn)
    test_loader = DataLoader(test_ds, batch_size=OCR_BATCH_SIZE, shuffle=False, collate_fn=ocr_collate_fn)

    print("Train:", len(train_ds), "Val:", len(val_ds), "Test:", len(test_ds))

## 20. OCR samples

In [ ]:

def show_ocr_samples(df, base_dir, n=9):
    if len(df) == 0:
        print("No OCR rows available.")
        return
    sample = df.sample(min(n, len(df)), random_state=42)
    cols = 3
    rows = math.ceil(len(sample) / cols)
    plt.figure(figsize=(14, 4 * rows))
    for i, (_, row) in enumerate(sample.iterrows(), 1):
        img_path = Path(base_dir) / row["split"] / row["image_name"]
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        plt.subplot(rows, cols, i)
        plt.imshow(img, cmap="gray")
        plt.title(row["text"])
        plt.axis("off")
    plt.tight_layout()
    plt.show()

if completed_labels_path.exists():
    show_ocr_samples(ocr_df, OCR_DIR / "images", n=9)

## 21. OCR model from scratch (CRNN)

In [ ]:

class CRNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d((2, 2)),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d((2, 2)),
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1), nn.ReLU(), nn.MaxPool2d((2, 1)),
            nn.Conv2d(256, 512, 3, padding=1), nn.ReLU(),
            nn.BatchNorm2d(512),
            nn.Conv2d(512, 512, 3, padding=1), nn.ReLU(),
            nn.BatchNorm2d(512),
            nn.MaxPool2d((2, 1)),
            nn.Conv2d(512, 512, 2), nn.ReLU()
        )
        self.rnn_input_size = 512 * 3
        self.rnn = nn.LSTM(
            input_size=self.rnn_input_size,
            hidden_size=256,
            num_layers=2,
            bidirectional=True,
            batch_first=False
        )
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.cnn(x)
        b, c, h, w = x.shape
        x = x.permute(3, 0, 1, 2)
        x = x.contiguous().view(w, b, c * h)
        x, _ = self.rnn(x)
        x = self.fc(x)
        return x

def ctc_greedy_decode(logits):
    pred = logits.argmax(dim=2).permute(1, 0)
    decoded = []
    for seq in pred:
        prev = -1
        chars = []
        for idx in seq.cpu().numpy():
            if idx != prev and idx != BLANK_IDX:
                chars.append(IDX_TO_CHAR.get(int(idx), ""))
            prev = idx
        decoded.append("".join(chars))
    return decoded

NUM_CLASSES = len(CHAR_SET) + 1
ocr_model = CRNN(NUM_CLASSES).to(DEVICE)
print(ocr_model)

## 22. OCR training helpers

In [ ]:

ctc_loss_fn = nn.CTCLoss(blank=BLANK_IDX, zero_infinity=True)
optimizer = optim.Adam(ocr_model.parameters(), lr=OCR_LR)

def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0.0
    for images, targets_concat, target_lengths, _ in tqdm(loader, desc="Train", leave=False):
        images = images.to(device)
        targets_concat = targets_concat.to(device)
        target_lengths = target_lengths.to(device)

        optimizer.zero_grad()
        logits = model(images)
        log_probs = logits.log_softmax(2)
        T, B, C = log_probs.shape
        input_lengths = torch.full((B,), T, dtype=torch.long).to(device)
        loss = loss_fn(log_probs, targets_concat, input_lengths, target_lengths)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / max(len(loader), 1)

def evaluate_ocr(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_targets = [], []
    with torch.no_grad():
        for images, targets_concat, target_lengths, target_texts in tqdm(loader, desc="Val/Test", leave=False):
            images = images.to(device)
            targets_concat = targets_concat.to(device)
            target_lengths = target_lengths.to(device)

            logits = model(images)
            log_probs = logits.log_softmax(2)
            T, B, C = log_probs.shape
            input_lengths = torch.full((B,), T, dtype=torch.long).to(device)
            loss = loss_fn(log_probs, targets_concat, input_lengths, target_lengths)
            total_loss += loss.item()

            preds = ctc_greedy_decode(logits)
            all_preds.extend(preds)
            all_targets.extend(target_texts)

    full_match = sum(p == g for p, g in zip(all_preds, all_targets))
    full_acc = full_match / max(len(all_targets), 1)

    char_match, char_total = 0, 0
    for pred, gt in zip(all_preds, all_targets):
        m = min(len(pred), len(gt))
        char_match += sum(pred[i] == gt[i] for i in range(m))
        char_total += max(len(gt), 1)
    char_acc = char_match / max(char_total, 1)

    return total_loss / max(len(loader), 1), full_acc, char_acc, all_preds, all_targets

## 23. Train OCR

In [ ]:

if completed_labels_path.exists():
    best_ocr_path = RUNS_DIR / "best_ocr_model.pth"
    best_val_loss = float("inf")
    history = []

    for epoch in range(1, OCR_EPOCHS + 1):
        train_loss = train_one_epoch(ocr_model, train_loader, optimizer, ctc_loss_fn, DEVICE)
        val_loss, val_full_acc, val_char_acc, _, _ = evaluate_ocr(ocr_model, val_loader, ctc_loss_fn, DEVICE)

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_full_acc": val_full_acc,
            "val_char_acc": val_char_acc
        })

        print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
              f"val_full_acc={val_full_acc:.4f} | val_char_acc={val_char_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(ocr_model.state_dict(), best_ocr_path)
            print("Saved best OCR model:", best_ocr_path)

    history_df = pd.DataFrame(history)
    display(history_df.tail())

## 24. Evaluate OCR

In [ ]:

if completed_labels_path.exists():
    best_ocr_path = RUNS_DIR / "best_ocr_model.pth"
    ocr_model.load_state_dict(torch.load(best_ocr_path, map_location=DEVICE))
    test_loss, test_full_acc, test_char_acc, test_preds, test_targets = evaluate_ocr(
        ocr_model, test_loader, ctc_loss_fn, DEVICE
    )
    print("Test loss:", round(test_loss, 4))
    print("Test full plate accuracy:", round(test_full_acc, 4))
    print("Test character accuracy:", round(test_char_acc, 4))

    pred_df = pd.DataFrame({"ground_truth": test_targets, "prediction": test_preds})
    display(pred_df.head(20))

# End-to-end inference

In [ ]:

def preprocess_plate_for_ocr(crop_bgr, img_w=OCR_IMG_W, img_h=OCR_IMG_H):
    gray = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, (img_w, img_h))
    gray = gray.astype(np.float32) / 255.0
    tensor = torch.tensor(gray[None, None, :, :], dtype=torch.float32).to(DEVICE)
    return tensor

def read_plate_text(crop_bgr, ocr_model):
    tensor = preprocess_plate_for_ocr(crop_bgr)
    with torch.no_grad():
        logits = ocr_model(tensor)
        return ctc_greedy_decode(logits)[0]

def detect_and_read_plate(image_path, detector_model, ocr_model=None, conf=0.25):
    img_bgr = cv2.imread(str(image_path))
    if img_bgr is None:
        raise FileNotFoundError(f"Could not read: {image_path}")

    results = detector_model(str(image_path), conf=conf)
    boxes = results[0].boxes
    vis = cv2.cvtColor(img_bgr.copy(), cv2.COLOR_BGR2RGB)
    preds = []

    if boxes is None or len(boxes) == 0:
        return vis, preds

    for box in boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
        score = float(box.conf[0].cpu().numpy())
        crop = img_bgr[max(0, y1):max(0, y2), max(0, x1):max(0, x2)]
        text = ""
        if ocr_model is not None and crop.size > 0:
            text = read_plate_text(crop, ocr_model)
        preds.append({"bbox": (x1, y1, x2, y2), "score": score, "text": text})
        cv2.rectangle(vis, (x1, y1), (x2, y2), (255, 0, 0), 2)
        label = f"{text} ({score:.2f})" if text else f"plate ({score:.2f})"
        cv2.putText(vis, label, (x1, max(15, y1 - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)
    return vis, preds

## 25. Predict on test images

In [ ]:

best_detector_path = RUNS_DIR / "indian_plate_detector" / "weights" / "best.pt"
detector_model = YOLO(str(best_detector_path))

ocr_model_for_infer = None
best_ocr_path = RUNS_DIR / "best_ocr_model.pth"
if best_ocr_path.exists():
    ocr_model_for_infer = CRNN(len(CHAR_SET) + 1).to(DEVICE)
    ocr_model_for_infer.load_state_dict(torch.load(best_ocr_path, map_location=DEVICE))
    ocr_model_for_infer.eval()

sample_imgs = list((DETECTOR_DIR / "images/test").glob("*.*"))[:6]

plt.figure(figsize=(16, 10))
for i, img_path in enumerate(sample_imgs, 1):
    vis, preds = detect_and_read_plate(img_path, detector_model, ocr_model_for_infer, conf=0.25)
    plt.subplot(2, 3, i)
    plt.imshow(vis)
    title = img_path.name
    texts = [p["text"] for p in preds if p["text"]]
    if len(texts) > 0:
        title += "\n" + ", ".join(texts)
    plt.title(title)
    plt.axis("off")
plt.tight_layout()
plt.show()

## 26. Single-image prediction

In [ ]:

CUSTOM_IMAGE = None  # example: Path("/content/my_car.jpg")

if CUSTOM_IMAGE is not None:
    vis, preds = detect_and_read_plate(CUSTOM_IMAGE, detector_model, ocr_model_for_infer, conf=0.25)
    plt.figure(figsize=(12, 8))
    plt.imshow(vis)
    plt.axis("off")
    plt.show()
    print("Predictions:")
    for p in preds:
        print(p)
else:
    print("Set CUSTOM_IMAGE to a real image path and rerun.")

## 27. Export predictions to CSV

In [ ]:

def batch_predict_and_save(image_dir, output_csv, detector_model, ocr_model=None):
    rows = []
    image_dir = Path(image_dir)
    for img_path in tqdm(list(image_dir.glob("*.*"))):
        _, preds = detect_and_read_plate(img_path, detector_model, ocr_model, conf=0.25)
        if len(preds) == 0:
            rows.append({"image_name": img_path.name, "detected": False, "plate_text": ""})
        else:
            best_pred = sorted(preds, key=lambda x: x["score"], reverse=True)[0]
            rows.append({
                "image_name": img_path.name,
                "detected": True,
                "plate_text": best_pred["text"]
            })
    out_df = pd.DataFrame(rows)
    out_df.to_csv(output_csv, index=False)
    return out_df

# Example:
# pred_df = batch_predict_and_save(DETECTOR_DIR / "images/test",
#                                  RUNS_DIR / "plate_predictions.csv",
#                                  detector_model,
#                                  ocr_model_for_infer)
# display(pred_df.head())

## Final note

This notebook is customized for the DataCluster Labs Indian Number Plates dataset family and is designed to work whether your extracted package contains YOLO, COCO, or Pascal VOC annotations.

If your exact download has a slightly different folder structure, the exploration and auto-detection cells help you adjust it quickly.